# Trainen voor de Mystery Device

In [23]:
import numpy as np
from tensorflow.keras.datasets import mnist

from mysterydevice_model import NeuralNetworkTraining, NeuralNetworkInference, sparse_cross_entropy
from read_image import load_image, classify_image, hidden_layer_sizes, output_nodes

## Definiëren van het model en functies

Voordat we het eindprogramma kunnen schrijven, moeten we natuurlijk eerst een model trainen. We hebben al verschillende modellen getraind in de P opdrachten, maar nu moeten we een model kiezen dat goed presteert en binnen de constraints van de Mystery Device past.

Zie `experimenten.ipynb` voor een overzicht van de gemaakte P opdrachten en de resultaten daarvan. Hier wordt ook de keuze voor het eindmodel gemotiveerd.

### Data laden

In [24]:
def load_mnist(npz_path='mnist.npz'):
    (x_train, y_train), (x_test, y_test) = mnist.load_data(path=npz_path)
    return (x_train, y_train), (x_test, y_test)

### Preprocessing

In [25]:
def normalize_data(x):
    return x / 255.0


def flatten_data(x):
    return x.reshape(len(x), -1)

### Model

In [26]:
input_nodes = 28 * 28
output_nodes = 10
dropout_rates = [0.2]
total_models = 12

hidden_layer_sizes = [[110 + (i * 3)] for i in range(total_models)]
subset_size = 0.7

## Trainen van het model

### Inladen en preprocessen van de data

In [27]:
(x_train, y_train), (x_test, y_test) = load_mnist()
x_train = normalize_data(flatten_data(x_train))
x_test = normalize_data(flatten_data(x_test))

### Trainen

In [28]:
from pathlib import Path

npz_files = list(Path(".").glob("weights*.npz"))
total_currently_saved_npz = len(npz_files)

if total_currently_saved_npz != total_models:
    for f in npz_files:
        f.unlink()

    for i in range(total_models):
        print(f"{'=' * 15} Model {i} {'=' * 15}")

        current_hidden_layer_sizes = hidden_layer_sizes[i]

        # Create a unique subset of the dataset
        sample_size = int(len(x_train) * subset_size)

        indices = np.random.choice(len(x_train), size=sample_size, replace=True)
        x_randomized = x_train[indices]
        y_randomized = y_train[indices]

        model = NeuralNetworkTraining(input_nodes, current_hidden_layer_sizes, output_nodes, dropout_rates)
        model.train(x_randomized, y_randomized, learning_rate=0.003, epochs=200, batch_size=16, val_split=0.1, patience=20, verbose=True)
        model.save_weights(f'weights_{i}.npz')

        if i < total_models - 1:
            print(f"\n\n")

=============== Model 0 ===============
Epoch   1 | Acc: 0.8838 | Loss: 0.4026 | Val Loss: 0.2086
Epoch   2 | Acc: 0.9406 | Loss: 0.2079 | Val Loss: 0.1400
Epoch   3 | Acc: 0.9535 | Loss: 0.1589 | Val Loss: 0.1169
Epoch   4 | Acc: 0.9635 | Loss: 0.1291 | Val Loss: 0.1024
Epoch   5 | Acc: 0.9681 | Loss: 0.1099 | Val Loss: 0.0904
Epoch   6 | Acc: 0.9711 | Loss: 0.0981 | Val Loss: 0.0868
Epoch   7 | Acc: 0.9751 | Loss: 0.0862 | Val Loss: 0.0777
Epoch   8 | Acc: 0.9773 | Loss: 0.0781 | Val Loss: 0.0754
Epoch   9 | Acc: 0.9797 | Loss: 0.0709 | Val Loss: 0.0721
Epoch  10 | Acc: 0.9810 | Loss: 0.0643 | Val Loss: 0.0692
Epoch  11 | Acc: 0.9826 | Loss: 0.0594 | Val Loss: 0.0658
Epoch  12 | Acc: 0.9839 | Loss: 0.0567 | Val Loss: 0.0623
Epoch  13 | Acc: 0.9852 | Loss: 0.0531 | Val Loss: 0.0616
Epoch  14 | Acc: 0.9861 | Loss: 0.0487 | Val Loss: 0.0598
Epoch  15 | Acc: 0.9868 | Loss: 0.0443 | Val Loss: 0.0615
Epoch  16 | Acc: 0.9874 | Loss: 0.0425 | Val Loss: 0.0580
Epoch  17 | Acc: 0.9872 | Loss: 

### Test de Models samen

In [29]:
all_probs = []

for i in range(total_models):
    current_hidden_layer_sizes = hidden_layer_sizes[i]

    m = NeuralNetworkInference(f'weights_{i}.npz')
    probs = m.forward(x_test)
    all_probs.append(probs.copy())
    del m

avg_probs = np.mean(all_probs, axis=0)
predicted = np.argmax(avg_probs, axis=1)
acc = np.mean(predicted == y_test)
loss = sparse_cross_entropy(y_test, avg_probs)

print("All Models:")
print(f"Test Accuracy {acc:.4f} | Test Loss: {loss:.4f}")

All Models:
Test Accuracy 0.9821 | Test Loss: 0.0591


### Peak RAM tijdens inference meten

Tijdens inference zijn er meerdere sources die deel uitmaken van het uiteindelijke totaal peak RAM tijdens inference.
Hieronder volgen de verschillende arrays die samen de peak RAM maken.

- Persistente arrays: altijd in RAM
  - Model weights
  - Model biases
  - Min/max per laag
- Tijdelijke arrays tijdens een forward pass
  - Input laag
  - Pre-activation
  - Current
  - Gedequantiseerde gewichten per kolom per laag
  - Output pre-activation
  - Prediction

In [30]:
max_model_idx = np.argmax([h[0] for h in hidden_layer_sizes])
worst_case_hidden_nodes = hidden_layer_sizes[max_model_idx][0]

_m = NeuralNetworkInference(f'weights_{max_model_idx}.npz')

persistent_weights = sum(w.nbytes for w in _m.weights)   # uint8
persistent_biases = sum(b.nbytes for b in _m.biases)     # float32
persistent_minmax = len(_m.w_min) * 2 * 4                # float32 min+max per laag

# _matmul dequantiseert kolom voor kolom; peak = één kolom van de grootste laag (laag 1: 784 floats)
peak_dequantized_col = input_nodes * 4

temporary = (
    1 * input_nodes * 4 +
    1 * worst_case_hidden_nodes * 4 +                    # preac laag 1
    1 * worst_case_hidden_nodes * 4 +                    # current laag 1
    peak_dequantized_col +                               # dequantized gewichtenkolom (peak: laag 1)
    1 * output_nodes * 4 +
    1 * output_nodes * 4
)

persistent = persistent_weights + persistent_biases + persistent_minmax

print(f"Worst-case Model Index:    {max_model_idx} ({worst_case_hidden_nodes} hidden nodes)")
print(f"Gewichten (uint8):         {persistent_weights / 1024:.2f} KB")
print(f"Biases (float32):          {persistent_biases / 1024:.2f} KB")
print(f"Min/max (float32):         {persistent_minmax / 1024:.2f} KB")
print(f"Tijdelijke arrays:         {temporary / 1024:.2f} KB")
print(f"Totaal peak:               {(persistent + temporary) / 1024:.2f} KB")

del _m

Worst-case Model Index:    11 (143 hidden nodes)
Gewichten (uint8):         110.88 KB
Biases (float32):          0.60 KB
Min/max (float32):         0.02 KB
Tijdelijke arrays:         7.32 KB
Totaal peak:               118.81 KB


### Peak RAM tijdens inference meten met tracemalloc

In [31]:
import tracemalloc

tracemalloc.start()

pre = tracemalloc.take_snapshot()
m = NeuralNetworkInference(f'weights_{max_model_idx}.npz')
snapshot_loaded = tracemalloc.take_snapshot()

test_image = load_image("0.png")

classify_image(test_image)

snapshot_predicted = tracemalloc.take_snapshot()

tracemalloc.stop()
del m

def total_kb(snapshot):
    return sum(stat.size for stat in snapshot.statistics('lineno')) / 1024

print(f"Begin runtime:   {total_kb(pre):.2f} KB")
print(f"Na laden model:  {total_kb(snapshot_loaded):.2f} KB")
print(f"Na predict:      {total_kb(snapshot_predicted):.2f} KB")

Begin runtime:   0.89 KB
Na laden model:  123.30 KB
Na predict:      124.83 KB


> De bovenstaande meting met tracemalloc geeft een indicatie van het RAM-gebruik tijdens verschillende fasen van de runtime.
> - Na het laden van het model zien we een aanzienlijke toename in RAM-gebruik, wat overeenkomt met de opslag van de modelgewichten, biases en min/max waarden.
>   - Het model zou nooit permanent in RAM staan zoals ik hier meet, maar dit geeft een indicatie van wat er in RAM ongeveer gebeurt aan het begin van `classify_image()`.
> - Na het uitvoeren van `classify_image()` zien we een verdere toename in RAM-gebruik, wat overeenkomt met de tijdelijke arrays die worden aangemaakt tijdens de forward pass.
> - In `classify_image()` worden de modellen met gc verwijderd na gebruik, zodat ze niet lui in RAM blijven staan.

> In de initiële run zal de "Na predict" waarde waarschijnlijk hoger liggen dan 256KB. Dit is te verwachten, gezien dit de uiteindelijke opstart van Python en de interne libraries betreft. Volgens de opdrachtbeschrijving hoeft hier geen rekening mee gehouden worden. Zodra de cel voor een tweede keer gerund wordt zonder de kernel te herstarten, is het in lijn met de waarde van "Na laden model" en daarom representatiever voor het daadwerkelijke geheugengebruik van het model.

### Totale opslag meten

In [32]:
from pathlib import Path

npz_files = sorted(Path(".").glob("weights*.npz"))

total = 0
for f in npz_files:
    kb = f.stat().st_size / 1024
    total += kb
    print(f"{f.name:<25} {kb:>8.2f} KB")

print("-" * 37)
print(f"{'Totaal':<25} {total:>8.2f} KB")

if npz_files:
    gemiddelde = total / len(npz_files)
    print(f"{'Gemiddelde':<25} {gemiddelde:>8.2f} KB")

print(f"{'Percentage van 1024 KB':<25} {total / 1024 * 100:>7.1f}%")

weights_0.npz                68.00 KB
weights_1.npz                71.41 KB
weights_10.npz               87.78 KB
weights_11.npz               90.37 KB
weights_2.npz                72.42 KB
weights_3.npz                76.00 KB
weights_4.npz                75.78 KB
weights_5.npz                79.41 KB
weights_6.npz                80.97 KB
weights_7.npz                82.46 KB
weights_8.npz                83.78 KB
weights_9.npz                85.82 KB
-------------------------------------
Totaal                      954.20 KB
Gemiddelde                   79.52 KB
Percentage van 1024 KB       93.2%


### Gemiddelde herkenningstijd van 1 image

In [33]:
import time
import gc

gc.freeze()

test_image = load_image("0.png")

runs = 100
start = time.perf_counter()
for _ in range(runs):
    classify_image(test_image)
end = time.perf_counter()

gc.unfreeze()

avg_ms = (end - start) / runs * 1000
print(f"Gemiddelde tijd per classificatie: {avg_ms:.2f} ms (over {runs} runs)")

Gemiddelde tijd per classificatie: 8.92 ms (over 100 runs)


> Goed om te weten: De TensorFlow import (al is het enkel de dataset) gooit veel in de garbage collector. Hierdoor wordt de gemiddelde tijd per classificatie aanzienlijk hoger dan de productie environment zou zijn. Om deze reden heb ik `gc.freeze()` en `gc.unfreeze()` neergezet bij de loops; deze zorgen er namelijk voor dat `gc.collect()` in de `classify_image()` functie niet meer de TensorFlow objecten hoeft te scannen. De modellen zelf worden nog wel netjes opgeruimd, maar de overhead van TensorFlow's import is niet meer aanwezig voor deze test. Hierdoor kan de meting dus accurater aantonen wat de daadwerkelijke performance zou zijn op de MysteryDevice.